# HFA AI Training - Module 3: Vector Databases
## Working with Pinecone

This notebook demonstrates the full Pinecone workflow:
1. Creating an index (like a table in a regular database)
2. Adding documents with embeddings and metadata
3. Querying by semantic similarity
4. Filtering results with metadata
5. Updating and deleting documents

Pinecone is a managed cloud vector database designed for production-scale similarity search. Unlike ChromaDB (which runs locally and auto-embeds), Pinecone requires explicit embeddings -- so we'll use OpenAI's `text-embedding-3-small` model to generate them.

### Install Dependencies

In [ ]:
!pip install pinecone openai

### Section 1: Initialize Pinecone

Pinecone is a managed cloud vector database. You need two API keys:
- **`PINECONE_API_KEY`** -- for accessing the Pinecone service
- **`OPENAI_API_KEY`** -- for generating embeddings (since Pinecone doesn't auto-embed)

Unlike ChromaDB which runs locally and auto-generates embeddings, Pinecone requires you to provide your own embedding vectors. We'll use OpenAI's `text-embedding-3-small` model for this.

In [ ]:
import os
import time
from pinecone import Pinecone, ServerlessSpec
from openai import OpenAI

# ============================================================
# SECTION 1: Initialize Pinecone
# ============================================================
# Pinecone is a managed cloud vector database.
# You need two API keys:
#   - PINECONE_API_KEY for the Pinecone service
#   - OPENAI_API_KEY for generating embeddings
#
# Unlike ChromaDB (which auto-embeds), Pinecone requires
# explicit embedding vectors, so we use OpenAI for that.

print("=" * 60)
print("HFA AI Training - Pinecone Example")
print("=" * 60)

# Initialize Pinecone client
pc = Pinecone(api_key=os.environ.get("PINECONE_API_KEY"))

# Initialize OpenAI client for generating embeddings
openai_client = OpenAI()

print("\nPinecone client initialized")
print("OpenAI client initialized (for embeddings)")

### Embedding Helper

Since Pinecone doesn't auto-embed documents like ChromaDB does, we need a helper function to generate embeddings using OpenAI's `text-embedding-3-small` model. This function will be used throughout the notebook whenever we add documents or run queries.

In [ ]:
# ============================================================
# EMBEDDING HELPER
# ============================================================
# Pinecone stores raw vectors — it doesn't generate embeddings
# for you. We'll use OpenAI's text-embedding-3-small model.

EMBEDDING_MODEL = "text-embedding-3-small"
EMBEDDING_DIM = 1536

def get_embeddings(texts: list[str]) -> list[list[float]]:
    """Generate embeddings using OpenAI."""
    response = openai_client.embeddings.create(
        model=EMBEDDING_MODEL,
        input=texts
    )
    return [item.embedding for item in response.data]

print(f"Embedding helper ready (model: {EMBEDDING_MODEL}, dimensions: {EMBEDDING_DIM})")

### Section 2: Create an Index

An "index" in Pinecone is like a table in a regular database (equivalent to a "collection" in ChromaDB). Each index stores vectors and their associated metadata.

Key differences from ChromaDB:
- You must specify the **dimension** of your vectors (1536 for `text-embedding-3-small`)
- You must specify a **metric** (cosine, euclidean, or dotproduct)
- You choose a **cloud provider and region** for the serverless deployment

In [ ]:
# ============================================================
# SECTION 2: Create an Index
# ============================================================
# An "index" in Pinecone is like a "collection" in ChromaDB.
# Each index stores vectors and their associated metadata.
#
# You must specify:
#   - dimension: size of embedding vectors (1536 for text-embedding-3-small)
#   - metric: similarity metric (cosine, euclidean, or dotproduct)
#   - spec: cloud provider and region for serverless deployment

INDEX_NAME = "hawaii-listings"

# Delete if exists (for clean re-runs)
if INDEX_NAME in [idx.name for idx in pc.list_indexes()]:
    pc.delete_index(INDEX_NAME)

pc.create_index(
    name=INDEX_NAME,
    dimension=EMBEDDING_DIM,
    metric="cosine",
    spec=ServerlessSpec(cloud="aws", region="us-east-1")
)

# Wait for the index to be ready
time.sleep(1)

index = pc.Index(INDEX_NAME)

print(f"\nCreated index: '{INDEX_NAME}'")
print(f"Dimension: {EMBEDDING_DIM}, Metric: cosine")
stats = index.describe_index_stats()
print(f"Current vector count: {stats.total_vector_count}")

### Section 3: Add Documents

Each document needs:
- **id**: A unique identifier (string)
- **values**: The embedding vector (we generate this with OpenAI)
- **metadata**: Key-value pairs for filtering (optional but useful)

Unlike ChromaDB (which auto-embeds your text), Pinecone requires you to provide pre-computed embedding vectors. We also store the document text in the metadata under a `"text"` key, since Pinecone only stores vectors and metadata -- it has no dedicated document text field.

In [ ]:
# ============================================================
# SECTION 3: Add Documents
# ============================================================
# Each document needs:
#   - id:       A unique identifier (string)
#   - values:   The embedding vector (generated via OpenAI)
#   - metadata: Key-value pairs for filtering (optional but useful)
#
# Unlike ChromaDB which auto-embeds, Pinecone requires explicit
# embedding vectors. We store document text in metadata under
# a "text" key since Pinecone has no dedicated document field.

print("\n--- Adding Documents ---\n")

# Real estate listings as documents
# Notice how each listing uses slightly different language
ids = [
    "listing_001",
    "listing_002",
    "listing_003",
    "listing_004",
    "listing_005",
    "listing_006",
    "listing_007",
    "listing_008",
    "listing_009",
    "listing_010",
]

documents = [
    "Beautiful 3-bedroom oceanfront home in Kailua with panoramic views of the Pacific. "
    "Newly renovated kitchen, hardwood floors throughout. Steps from the beach.",

    "Charming 2-bed cottage in Haleiwa on the North Shore. Surf breaks within walking "
    "distance. Tropical garden, outdoor shower, laid-back island living at its finest.",

    "Luxury 4-bedroom penthouse in Waikiki with floor-to-ceiling windows overlooking "
    "Diamond Head. Full concierge service, rooftop pool, premium finishes throughout.",

    "Affordable 3-bed starter home in Ewa Beach. Family-friendly neighborhood, close "
    "to schools and shopping. Large backyard, perfect for growing families.",

    "Secluded mountain retreat in Manoa Valley. 2-bedroom home surrounded by lush "
    "tropical rainforest. Private trail access, serene waterfall nearby. Total privacy.",

    "Modern 1-bedroom condo in Kakaako with city and ocean views. Walking distance "
    "to restaurants, shops, and Ala Moana Beach Park. Ideal for young professionals.",

    "Historic plantation-style estate in Kailua-Kona on the Big Island. 5 bedrooms, "
    "wraparound lanai, mango and avocado trees. Rich with Hawaiian heritage.",

    "Beachfront studio in Maui with direct sand access. Perfect vacation rental or "
    "island getaway. Fully furnished, turnkey investment property.",

    "Spacious 4-bed family home in Mililani with mountain views. Excellent schools "
    "nearby, community pool, parks, and walking trails. Quiet, safe neighborhood.",

    "Eco-friendly off-grid tiny home on 2 acres in Puna, Big Island. Solar powered, "
    "rainwater catchment, organic garden. Sustainable Hawaiian living.",
]

metadatas = [
    {"beds": 3, "area": "Kailua", "island": "Oahu", "type": "house", "price_range": "high"},
    {"beds": 2, "area": "Haleiwa", "island": "Oahu", "type": "cottage", "price_range": "medium"},
    {"beds": 4, "area": "Waikiki", "island": "Oahu", "type": "penthouse", "price_range": "luxury"},
    {"beds": 3, "area": "Ewa Beach", "island": "Oahu", "type": "house", "price_range": "low"},
    {"beds": 2, "area": "Manoa", "island": "Oahu", "type": "house", "price_range": "high"},
    {"beds": 1, "area": "Kakaako", "island": "Oahu", "type": "condo", "price_range": "medium"},
    {"beds": 5, "area": "Kailua-Kona", "island": "Big Island", "type": "estate", "price_range": "high"},
    {"beds": 1, "area": "Maui", "island": "Maui", "type": "studio", "price_range": "medium"},
    {"beds": 4, "area": "Mililani", "island": "Oahu", "type": "house", "price_range": "medium"},
    {"beds": 1, "area": "Puna", "island": "Big Island", "type": "tiny home", "price_range": "low"},
]

# Generate embeddings for all documents using OpenAI
print("Generating embeddings via OpenAI...")
embeddings = get_embeddings(documents)
print(f"Generated {len(embeddings)} embeddings (dimension: {len(embeddings[0])})")

# Upsert to Pinecone
# We store the document text in metadata since Pinecone only stores vectors + metadata
vectors = []
for i, (doc_id, embedding, metadata) in enumerate(zip(ids, embeddings, metadatas)):
    meta = {**metadata, "text": documents[i]}
    vectors.append({"id": doc_id, "values": embedding, "metadata": meta})

index.upsert(vectors=vectors)

# Pinecone serverless indexes may take a moment to become queryable after upsert
time.sleep(2)

stats = index.describe_index_stats()
print(f"\nAdded {stats.total_vector_count} listings to the index.")
print("Embeddings were generated by OpenAI's text-embedding-3-small model.")

### Section 4: Basic Semantic Query

Now the magic: we can search using natural language, and Pinecone finds the most semantically similar documents.

We first embed the query text using the same OpenAI model, then send the embedding vector to Pinecone:
- **vector**: The query embedding
- **top_k**: How many results to return
- **include_metadata**: Whether to return metadata with results

Note: Pinecone cosine similarity scores are **higher = more similar** (unlike ChromaDB distances where lower = more similar).

In [ ]:
# ============================================================
# SECTION 4: Basic Semantic Query
# ============================================================
# Now the magic: we can search using natural language,
# and Pinecone finds the most semantically similar documents.
#
# We embed the query with OpenAI, then search Pinecone.
# Pinecone cosine scores: higher = more similar
# (opposite of ChromaDB distances where lower = more similar)

print("\n\n--- QUERY 1: Basic Semantic Search ---")
print('Query: "beach house perfect for a family vacation"\n')

# Embed the query using the same model
query_embedding = get_embeddings(["beach house perfect for a family vacation"])[0]

results = index.query(
    vector=query_embedding,
    top_k=3,
    include_metadata=True,
)

# Display results
for i, match in enumerate(results.matches):
    print(f"  Result #{i+1} (score: {match.score:.4f})")
    print(f"    ID:       {match.id}")
    print(f"    Area:     {match.metadata['area']}, {match.metadata['island']}")
    print(f"    Type:     {match.metadata['type']} | Beds: {match.metadata['beds']}")
    print(f"    Snippet:  {match.metadata['text'][:100]}...")
    print()

print("  Notice: It found beach-related family listings even though")
print("  the query words don't exactly match the listing text!")

### Section 5: Query with Metadata Filtering

The real power: combine semantic search with structured filters. "Find me something LIKE this (semantic) that ALSO meets these criteria (metadata filters)."

Pinecone supports: `$eq`, `$ne`, `$gt`, `$gte`, `$lt`, `$lte`, `$in`, `$nin`, combined with `$and`, `$or` -- very similar to ChromaDB's filter syntax.

In [ ]:
# ============================================================
# SECTION 5: Query with Metadata Filtering
# ============================================================
# The real power: combine semantic search with structured filters.
# "Find me something LIKE this (semantic) that ALSO meets these
#  criteria (metadata filters)."
#
# Pinecone supports: $eq, $ne, $gt, $gte, $lt, $lte, $in, $nin
# Combined with: $and, $or

print("\n--- QUERY 2: Semantic Search + Metadata Filtering ---")
print('Query: "quiet peaceful nature retreat"')
print('Filter: Oahu island only, at least 2 bedrooms\n')

query_embedding = get_embeddings(["quiet peaceful nature retreat"])[0]

results = index.query(
    vector=query_embedding,
    top_k=3,
    include_metadata=True,
    filter={
        "$and": [
            {"island": {"$eq": "Oahu"}},
            {"beds": {"$gte": 2}},
        ]
    },
)

for i, match in enumerate(results.matches):
    print(f"  Result #{i+1} (score: {match.score:.4f})")
    print(f"    ID:       {match.id}")
    print(f"    Area:     {match.metadata['area']}, {match.metadata['island']}")
    print(f"    Type:     {match.metadata['type']} | Beds: {match.metadata['beds']}")
    print(f"    Snippet:  {match.metadata['text'][:100]}...")
    print()

print("  The Manoa Valley retreat should rank highly — it matches the")
print("  'quiet peaceful nature' meaning AND the Oahu + 2-bed filters.")

### Section 6: Query with Price Range Filter

Let's filter for affordable options using the `$in` operator.

In [ ]:
# ============================================================
# SECTION 6: Query with Price Range Filter
# ============================================================
# Let's filter for affordable options

print("\n--- QUERY 3: Affordable Family Homes ---")
print('Query: "family home with good schools and outdoor space"')
print('Filter: Low or medium price range\n')

query_embedding = get_embeddings(["family home with good schools and outdoor space"])[0]

results = index.query(
    vector=query_embedding,
    top_k=3,
    include_metadata=True,
    filter={
        "price_range": {"$in": ["low", "medium"]},
    },
)

for i, match in enumerate(results.matches):
    print(f"  Result #{i+1} (score: {match.score:.4f})")
    print(f"    ID:       {match.id}")
    print(f"    Area:     {match.metadata['area']}, {match.metadata['island']}")
    print(f"    Type:     {match.metadata['type']} | Beds: {match.metadata['beds']} | Price: {match.metadata['price_range']}")
    print(f"    Snippet:  {match.metadata['text'][:100]}...")
    print()

### Section 7: Multiple Queries

Unlike ChromaDB which supports batch queries in a single call, Pinecone processes one query vector at a time. We run multiple queries in a loop.

In [ ]:
# ============================================================
# SECTION 7: Multiple Queries
# ============================================================
# Unlike ChromaDB which supports batch queries in a single call,
# Pinecone processes one query vector at a time.
# We run multiple queries in a loop.

print("\n--- QUERY 4: Multiple Queries (Loop) ---\n")

queries = [
    "surfing lifestyle near the waves",
    "luxury living with premium amenities",
    "sustainable eco-friendly living off the grid",
]

for query_text in queries:
    query_emb = get_embeddings([query_text])[0]
    results = index.query(
        vector=query_emb,
        top_k=2,
        include_metadata=True,
    )

    print(f'  Query: "{query_text}"')
    for match in results.matches:
        print(f"    -> {match.metadata['area']} {match.metadata['type']} (score: {match.score:.4f})")
    print()

### Section 8: Update a Document

You can update metadata for existing vectors. Here we'll simulate a price drop by updating metadata. In Pinecone, use `index.update()` with `set_metadata` to modify metadata fields without re-uploading the vector.

In [ ]:
# ============================================================
# SECTION 8: Update a Document
# ============================================================
# You can update metadata without re-uploading the vector

print("\n--- Updating a Document ---\n")

# Let's update the metadata for a listing (price drop!)
index.update(
    id="listing_004",
    set_metadata={"note": "PRICE REDUCED!", "price_range": "low"},
)

# Brief wait for the update to propagate
time.sleep(1)

# Verify the update by fetching the document directly
result = index.fetch(ids=["listing_004"])
print(f"  Updated listing_004 metadata: {result.vectors['listing_004'].metadata}")

### Section 9: Delete a Document

Vectors can be removed from an index by ID (e.g., when a listing is sold).

In [ ]:
# ============================================================
# SECTION 9: Delete a Document
# ============================================================

print("\n--- Deleting a Document ---\n")

stats_before = index.describe_index_stats()
print(f"  Vector count before delete: {stats_before.total_vector_count}")

# Remove a listing (maybe it sold!)
index.delete(ids=["listing_008"])

# Brief wait for delete to propagate
time.sleep(1)

stats_after = index.describe_index_stats()
print(f"  Vector count after delete:  {stats_after.total_vector_count}")
print("  (listing_008 - Maui studio - has been removed)")

### Section 10: Describe the Index

Pinecone's `describe_index_stats()` method shows summary information about the index -- useful for quick inspection. This is the equivalent of ChromaDB's `peek()` method.

In [ ]:
# ============================================================
# SECTION 10: Describe the Index
# ============================================================

print("\n--- Describing the Index ---\n")

stats = index.describe_index_stats()
print(f"  Index '{INDEX_NAME}' stats:")
print(f"  Total vectors: {stats.total_vector_count}")
print(f"  Dimension: {stats.dimension}")
print(f"  Namespaces: {dict(stats.namespaces)}")

### Key Takeaways

In [ ]:
# ============================================================
# SUMMARY
# ============================================================
print("\n" + "=" * 60)
print("KEY TAKEAWAYS:")
print("=" * 60)
print("""
  1. Pinecone is a managed cloud vector database — no local server needed.
  2. You provide your own embeddings (e.g., via OpenAI's text-embedding-3-small).
  3. Query with embedding vectors — results ranked by cosine similarity (higher = better).
  4. Combine semantic search with metadata filters for precise results.
  5. Supports updates, deletes, and index statistics.
  6. Pinecone is production-ready and scales automatically with serverless architecture.

  Key differences from ChromaDB:
  - ChromaDB auto-embeds; Pinecone requires explicit embeddings.
  - ChromaDB uses distances (lower = better); Pinecone uses similarity scores (higher = better).
  - ChromaDB runs locally; Pinecone is a managed cloud service.
  - Document text must be stored in Pinecone metadata (no dedicated document field).

  Next: See semantic_search.ipynb for a full real estate search demo!
""")

### Cleanup

Delete the Pinecone index to avoid ongoing charges. In a production app you'd keep the index running, but for this demo we clean up after ourselves.

In [ ]:
# ============================================================
# CLEANUP: Delete the Pinecone index
# ============================================================
# Delete the index when done to avoid ongoing charges

pc.delete_index(INDEX_NAME)
print(f"Deleted index '{INDEX_NAME}' — cleanup complete.")